### Phase 2: Feature Selection with simple methods

In [22]:
import pandas as pd
from data_preprocessing import create_train_test_val_sets, get_processed_df
import joblib
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [23]:
#Create test train splits
x_mendeley, y_mendeley = get_processed_df(r"..\data\raw\Mendeley Dataset.csv")
x_kaggle, y_kaggle= get_processed_df(r"..\data\raw\dataset_phishing.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
kaggle_sets = create_train_test_val_sets(x_kaggle,y_kaggle, label_col="Label", test_size=0.2, n_splits=5)

----------Processing None Dataset----------

Class Distribution:
Label
0    0.518415
1    0.481585
Name: proportion, dtype: float64
int64

Total Missing Values: 0
No categorical features to hash
Shape After Processing: (247950, 42)
True
----------Processing None Dataset----------

Class Distribution:
Label
legitimate    0.5
phishing      0.5
Name: proportion, dtype: float64
object

Total Missing Values: 0
Shape After Processing: (11430, 32856)
True
Train/validation/test split prepared: 210757 instances for training, 37193 instances for validation, 49590 instances for testing
Stratified 5-fold CV splits created.
Train/validation/test split prepared: 9715 instances for training, 1715 instances for validation, 2286 instances for testing
Stratified 5-fold CV splits created.


In [24]:
kaggle_sets["y_train"] = kaggle_sets["y_train"].astype(int)
kaggle_sets["y_val"] = kaggle_sets["y_val"].astype(int)
kaggle_sets["y_test"] = kaggle_sets["y_test"].astype(int)

In [25]:
from sklearn.feature_selection import mutual_info_classif
import matplotlib.pyplot as plt

#Mendeley
features = mendeley_sets["x_train"].columns
mi_mendeley = mutual_info_classif(mendeley_sets["x_train"], mendeley_sets["y_train"], random_state=42)
mi_df_mendeley = pd.Series(mi_mendeley, index=features)

#plot to see the best features
mi_df_mendeley.sort_values(ascending=False).plot.bar(figsize=(15, 7))
plt.title("Feature Importance using Mutual Information")
plt.show()


#Phiusiil
features = kaggle_sets["x_train"].columns
mi_phiusiil = mutual_info_classif(kaggle_sets["x_train"], kaggle_sets["y_train"], random_state=42)
mi_df_phusiil = pd.Series(mi_phiusiil, index=features)

#plot to see the best features
mi_df_phusiil[mi_df_phusiil > 0.1].sort_values(ascending=False).plot.bar(figsize=(15, 7))
plt.title("Feature Importance using Mutual Information")
plt.show()

KeyboardInterrupt: 

In [26]:
#import phase 1 models
xgb_mendeley = joblib.load('./models/phase_1/xgboost_mendeley_no_fs.joblib')
xgb_kaggle = joblib.load('./models/phase_1/xgboost_kaggle_no_fs.joblib')
logreg_mendeley = joblib.load('./models/phase_1/logreg_mendeley_no_fs.joblib')
logreg_kaggle = joblib.load('./models/phase_1/logreg_kaggle_no_fs.joblib')

In [30]:
#train models on reduced feature set
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.feature_selection import SelectKBest
from sklearn.base import clone

def get_k_values(dataset, model, values, dataset_name, model_name, selector_func, param_name):
    performance_history = []
    best_values = {"f1": 0, "k": 0}
    for val in values:
        model_clone = clone(model)
        selector = selector_func(val)
        selector.fit(dataset["x_train"], dataset["y_train"])

        X_train_selected = selector.transform(dataset["x_train"])
        X_val_selected = selector.transform(dataset["x_val"])

        try:
            selected_names = dataset["x_train"].columns[selector.get_support()]
        except:
            selected_names = None

        model_clone.fit(X_train_selected, dataset["y_train"])
        y_val_pred = model_clone.predict(X_val_selected)

        score = f1_score(dataset["y_val"], y_val_pred)
        print("Model =", model_name, param_name, "=", val, " f1_score =", score)
        
        if score > best_values["f1"]:
            best_values = {"f1": score, param_name: val}

        performance_history.append({
            'Dataset': dataset_name,
            'Model': model_name,
            param_name: val,
            'f1_Score': score,
            'Precision': precision_score(dataset["y_val"], y_val_pred),
            'Recall': recall_score(dataset["y_val"], y_val_pred),
            'Features': selected_names
        })
    return performance_history, best_values

mi_best_performance = []

#perform feature selection on mendeley dataset
k_values = [10, 15, 20, 25, 30, 35, 40]
print("=== Mendeley Results ===")
performance, best_mendeley_xgb = get_k_values(mendeley_sets, xgb_mendeley, k_values, "Mendeley", 'XGBoost', lambda v: SelectKBest(mutual_info_classif, k=v), "k")
mi_best_performance += performance    #concatenate list
performance, best_mendeley_logreg = get_k_values(mendeley_sets, logreg_mendeley, k_values, "Mendeley", 'LogReg', lambda v: SelectKBest(mutual_info_classif, k=v), "k")
mi_best_performance += performance    #concatenate list

#perform feature selection on the kaggle dataset
k_values = [5000, 10000, 15000, 20000, 25000, 30000]
print("=== Kaggle Results ===")
performance, best_kaggle_xgb = get_k_values(kaggle_sets, xgb_kaggle, k_values, "Kaggle", 'XGBoost', lambda v: SelectKBest(mutual_info_classif, k=v), "k")
mi_best_performance += performance    #concatenate list
performance, best_kaggle_logreg = get_k_values(kaggle_sets, logreg_kaggle, k_values, "Kaggle", 'LogReg', lambda v: SelectKBest(mutual_info_classif, k=v), "k")
mi_best_performance += performance    #concatenate list

#print results and save history
performance_df = pd.DataFrame(mi_best_performance)
os.makedirs("../results/phase_2", exist_ok=True) # Ensure directory exists
performance_df.to_csv("../results/phase_2/mi_feature_selection_results.csv", index=False)
print("\nResults saved to '../results/phase_2/mi_feature_selection_results.csv'")


=== Mendeley Results ===
Model = XGBoost k = 20  f1_score = 0.9442826855123675
Model = XGBoost k = 25  f1_score = 0.9441351832490322
Model = XGBoost k = 30  f1_score = 0.9447468622197152
Model = XGBoost k = 35  f1_score = 0.9446592352194159
Model = XGBoost k = 40  f1_score = 0.9441478091363675
Model = LogReg k = 20  f1_score = 0.7778961069695894
Model = LogReg k = 25  f1_score = 0.7797961896322552
Model = LogReg k = 30  f1_score = 0.784268534113406
Model = LogReg k = 35  f1_score = 0.784610383853905
Model = LogReg k = 40  f1_score = 0.7848340672005674
=== Kaggle Results ===
Model = XGBoost k = 20000  f1_score = 0.9697674418604652
Model = XGBoost k = 25000  f1_score = 0.971561230412072
Model = XGBoost k = 30000  f1_score = 0.971561230412072
Model = LogReg k = 20000  f1_score = 0.9655172413793104
Model = LogReg k = 25000  f1_score = 0.9672514619883041
Model = LogReg k = 30000  f1_score = 0.9666861484511982

Results saved to '../results/phase_2/mi_feature_selection_results.csv'


In [ ]:
phase_2_results = []
for ds_name, model_name, best_info, dataset, model, method in [
    ("Mendeley", "XGBoost", best_mendeley_xgb, mendeley_sets, xgb_mendeley, "Mutual Info"),
    ("Kaggle", "XGBoost", best_kaggle_xgb, kaggle_sets, xgb_kaggle, "Mutual Info"),
    ("Mendeley", "LogReg", best_mendeley_logreg, mendeley_sets, logreg_mendeley, "Mutual Info"),
    ("Kaggle", "LogReg", best_kaggle_logreg, kaggle_sets, logreg_kaggle, "Mutual Info")
]:
    #combine train + val
    X_full = pd.concat([dataset["x_train"], dataset["x_val"]])
    y_full = pd.concat([dataset["y_train"], dataset["y_val"]])

    selector = SelectKBest(mutual_info_classif, k=best_info['k'])
    selector.fit(X_full, y_full)

    X_selected = selector.transform(X_full)
    X_test_selected = selector.transform(dataset["x_test"])

    selected_features = dataset["x_train"].columns[selector.get_support()]

    model.fit(X_selected, y_full)
    y_test_pred = model.predict(X_test_selected)
    
    #Final Metrics
    f1 = f1_score(dataset["y_test"], y_test_pred)
    prec = precision_score(dataset["y_test"], y_test_pred)
    rec = recall_score(dataset["y_test"], y_test_pred)

    print(f" Best {model_name} for {ds_name} using {method}: fs_param={best_info['k']} | Test F1: {f1:.4f}")

    #Store metrics for the summary CSV
    phase_2_results.append({
        'Dataset': ds_name,
        'Model': model_name,
        'Method': method,
        'Best_K': best_info['k'],
        'Test_F1': f1,
        'Test_Precision': prec,
        'Test_Recall': rec,
        'Features_Used': selected_features
    })

    #Save Raw Predictions
    pred_df = pd.DataFrame({
        'Actual_Label': dataset["y_test"],
        'Predicted_Label': y_test_pred
    })
    pred_df.to_csv(f"../results/phase_2/{model_name}_{ds_name.lower()}_test_predictions_mi.csv", index=False)

#Write the Final Summary CSV
summary_df = pd.DataFrame(phase_2_results)
summary_df.to_csv("../results/phase_2/mi_fs_final_summary.csv", index=False)

 Best XGBoost Mendeley: k=35 | Test F1: 0.9685
 Best XGBoost Kaggle: k=25000 | Test F1: 1.0000
 Best LogReg Mendeley: k=40 | Test F1: 0.7850
 Best LogReg Kaggle: k=10000 | Test F1: 1.0000
